# HAM10000 Edge-AI Compression Pipeline: Colab Runner

Run cells top to bottom on a Colab GPU runtime (Runtime > Change runtime type > GPU). You need a Kaggle API token (`kaggle.json`).

Steps: install dependencies -> download HAM10000 -> clone this repo -> train baseline -> compress -> benchmark -> results table and plot.

In [ ]:
# 1. Install dependencies. tfmot needs legacy Keras 2, so set this before TensorFlow is imported.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
!pip install -q tensorflow-model-optimization tf-keras kaggle

In [ ]:
# 2. Upload your Kaggle API token (kaggle.json)
from google.colab import files
print("Upload kaggle.json (kaggle.com -> Account -> Create New API Token)")
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# 3. Download HAM10000
!mkdir -p /content/data/HAM10000
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data/HAM10000 --unzip

In [ ]:
# 4. Clone the project and point it at the dataset
!git clone https://github.com/vijay1011-stack11/edge-skin-lesion-compression.git
%cd edge-skin-lesion-compression
os.environ["HAM10000_DIR"] = "/content/data/HAM10000"

In [ ]:
# 5. Check the configuration (values are set in src/config.py)
from src import config
print("Data dir:", config.DATA_DIR)
print("Weights:", config.MOBILENET_WEIGHTS)
print("Unfreeze from layer:", config.FINETUNE_UNFREEZE_FROM, "| fine-tune epochs (max):", config.BASELINE_EPOCHS_FINETUNE)
print("Baseline class-weight boost:", config.BASELINE_CLASS_WEIGHT_BOOST)

In [ ]:
# 6. Train baseline (frozen head -> fine-tune). ~15-30 min on Colab GPU depending on tier.
from src import train_baseline
train_baseline.main()

In [ ]:
# 7. Compression sweep: PTQ (dynamic + int8), pruning at 30/50/70%, pruning+int8, QAT+int8
from src import compress
compress.main()

In [ ]:
# 8. Benchmark every .tflite variant: size, CPU latency, accuracy, per-class F1
from src import benchmark
benchmark.main()

In [ ]:
# 9. Inspect the Pareto table and plot accuracy vs. size vs. latency
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("reports/pareto_results.csv")
display(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(df["size_kb"], df["macro_f1"])
for _, r in df.iterrows():
    axes[0].annotate(r["model"], (r["size_kb"], r["macro_f1"]), fontsize=7)
axes[0].set_xlabel("Size (KB)"); axes[0].set_ylabel("Macro F1"); axes[0].set_title("Size vs. Macro F1")

axes[1].scatter(df["latency_ms"], df["macro_f1"])
for _, r in df.iterrows():
    axes[1].annotate(r["model"], (r["latency_ms"], r["macro_f1"]), fontsize=7)
axes[1].set_xlabel("CPU Latency (ms)"); axes[1].set_ylabel("Macro F1"); axes[1].set_title("Latency vs. Macro F1")
plt.tight_layout(); plt.savefig("reports/pareto_plot.png", dpi=150); plt.show()

# Critical-class (mel/akiec) degradation check
print(df[["model", "macro_f1", "f1_mel", "f1_akiec"]])

In [ ]:
# 10. Zip up models + reports so you can download everything back out of Colab
!zip -r results_bundle.zip models reports
from google.colab import files
files.download("results_bundle.zip")